In [ ]:
#@title ⚙️ نصب و دانلود (کتابخانه‌ها + مدل OmniVoice)

HF_TOKEN = "" #@param {type:"string"}
INSTALL_TEXT_NORMALIZER = True #@param {type:"boolean"}
ALLOW_CPU = False #@param {type:"boolean"}

import os, sys, base64, shutil, subprocess

if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()
    print('✅ توکن Hugging Face تنظیم شد.')
else:
    os.environ.pop('HF_TOKEN', None)
    print('ℹ️ بدون توکن Hugging Face ادامه می‌دهیم.')

print(base64.b64decode("Tm90ZWJvb2sgTWFkZSBCeSBhaWdvbGRlbg==").decode())
print('=' * 30)

# ── ۱) PyTorch و torchaudio با هم و از یک ایندکس
# (اگر torchaudio جدا/بعد نصب شود، ممکن است نسخه‌ی torch را عوض کند و ABI بشکند.
#  --index-url استفاده می‌شود تا نسخه‌ی CPU از PyPI به‌جای نسخه‌ی CUDA انتخاب نشود.)
print('⏳ نصب PyTorch + torchaudio (cu128) ...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'torch', 'torchaudio',
     '--index-url', 'https://download.pytorch.org/whl/cu128'],
)
if r.returncode != 0:
    print('⚠️ نصب از ایندکس cu128 ناموفق بود؛ تلاش با ایندکس پیش‌فرض PyPI (نسخه‌ی CPU).')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchaudio'], check=True)

# ── ۲) کتابخانه‌های کمکی — بدون -U روی ipywidgets تا ویجت‌های Colab نشکنند
print('⏳ نصب کتابخانه‌های کمکی ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'python-docx', 'pydub', 'soundfile', 'ipywidgets'],
    check=True,
)

# ── ۳) OmniVoice (با اکسترای نرمال‌سازی متن برای خواندن درست اعداد)
print('⏳ نصب OmniVoice ...')
pkg = 'omnivoice[tn]' if INSTALL_TEXT_NORMALIZER else 'omnivoice'
if subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg]).returncode != 0:
    if INSTALL_TEXT_NORMALIZER:
        print('⚠️ نصب omnivoice[tn] ناموفق بود؛ بدون نرمال‌ساز متن ادامه می‌دهیم.')
        INSTALL_TEXT_NORMALIZER = False
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'omnivoice'], check=True)

# ── ۴) ffmpeg
if shutil.which('ffmpeg') is None:
    print('⏳ نصب ffmpeg ...')
    subprocess.run('apt-get update -qq && apt-get install -y -qq ffmpeg',
                   shell=True, check=True)

import torch
from omnivoice import OmniVoice

if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError(
        '❌ GPU در دسترس نیست. اجرا روی CPU برای این مدل عملاً غیرقابل استفاده است.\n'
        '   در Colab: Runtime → Change runtime type → Hardware accelerator → T4 GPU\n'
        '   اگر واقعاً می‌خواهید روی CPU اجرا شود، ALLOW_CPU را True کنید.'
    )

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device.startswith('cuda') else torch.float32

print(f'Device: {device}')

if device.startswith('cuda'):
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} ({props.total_memory/1e9:.1f} GB)')
    if props.total_memory < 12e9:
        print('⚠️ حافظه‌ی GPU کم است؛ برای اسناد طولانی ممکن است با OOM مواجه شوید.')
else:
    print('⚠️ اجرا روی CPU — هر پاراگراف ممکن است چند دقیقه طول بکشد.')

print('در حال دانلود و لود مدل OmniVoice (اولین بار ~۳.۳ گیگابایت دانلود می‌کند)...')

omni_model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice',
    device_map=device,
    dtype=dtype,
)

print('✅ همه چیز آماده‌ست، مدل لود شد!')
if device.startswith('cuda'):
    used = torch.cuda.memory_allocated() / 1e9
    print(f'💾 حافظه‌ی اشغال‌شده‌ی GPU: {used:.2f} GB')


In [ ]:
#@title 📂 انتخاب فایل متنی (گوگل درایو یا حافظه داخلی)

import os

Use_Local_File = True #@param {type:"boolean"}

# فرمت‌های متنی که «خواندن با صدا» برایشان معنی دارد
TEXT_EXTENSIONS = (
    '.txt', '.docx', '.md', '.rtf',
    '.srt', '.vtt', '.sub', '.ass',
)

SELECTED_DOC_PATH = None

if Use_Local_File:

    from google.colab import files

    print('📁 لطفاً فایل متنی خود را انتخاب کنید...')
    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError('❌ فایلی انتخاب نشد!')

    fname = list(uploaded.keys())[0]

    if not fname.lower().endswith(TEXT_EXTENSIONS):
        raise RuntimeError(
            '❌ این فرمت پشتیبانی نمی‌شود!\n'
            f'فرمت‌های مجاز: {", ".join(TEXT_EXTENSIONS)}'
        )

    SELECTED_DOC_PATH = os.path.abspath(fname)
    print(f'✅ فایل انتخاب شد: {SELECTED_DOC_PATH}')

else:

    from google.colab import drive
    import ipywidgets as widgets
    from IPython.display import display

    print('🔄 در حال اتصال به Google Drive...')
    drive.mount('/content/drive', force_remount=True)

    ROOT_DIR = '/content/drive/MyDrive'
    current_dir = [ROOT_DIR]

    path_label = widgets.HTML()
    listing = widgets.Select(rows=14, layout=widgets.Layout(width='100%'))
    btn_open = widgets.Button(description='📂 باز کردن / تأیید انتخاب', button_style='primary')
    btn_up = widgets.Button(description='⬆️ پوشه بالاتر')
    status = widgets.HTML()

    def refresh_listing():
        path_label.value = f'<b>📍 مسیر فعلی:</b> {current_dir[0]}'
        try:
            entries = sorted(os.listdir(current_dir[0]), key=lambda x: x.lower())
        except Exception as e:
            status.value = f'<span style="color:red">❌ خطا در خواندن پوشه: {e}</span>'
            listing.options = []
            return

        options = []
        for e in entries:
            full = os.path.join(current_dir[0], e)
            if os.path.isdir(full):
                options.append(f'📁 {e}')
            elif e.lower().endswith(TEXT_EXTENSIONS):
                options.append(f'📄 {e}')

        listing.options = options
        status.value = ('' if options else
                        '<span style="color:#888">📭 فایل متنی پشتیبانی‌شده‌ای در این پوشه نیست.</span>')

    def on_up_clicked(b):
        if os.path.normpath(current_dir[0]) == os.path.normpath(ROOT_DIR):
            status.value = '<span style="color:orange">⚠️ شما در پوشه اصلی Drive هستید.</span>'
            return
        parent = os.path.dirname(current_dir[0].rstrip('/'))
        if parent and len(parent) >= len(ROOT_DIR):
            current_dir[0] = parent
            refresh_listing()

    def on_open_clicked(b):
        global SELECTED_DOC_PATH

        if not listing.value:
            status.value = '<span style="color:orange">⚠️ ابتدا یک فایل انتخاب کنید.</span>'
            return

        name = listing.value[2:].strip()
        full = os.path.join(current_dir[0], name)

        if os.path.isdir(full):
            current_dir[0] = full
            refresh_listing()
            return

        if full.lower().endswith(TEXT_EXTENSIONS):
            SELECTED_DOC_PATH = full
            status.value = ('<div style="margin-top:8px;padding:10px;border-radius:6px;'
                            'background:#e8f5e9;color:#1b5e20;font-weight:bold;">'
                            f'✅ فایل تأیید شد.<br>📄 {name}</div>')
            print(f'✅ فایل انتخاب شد:\n{SELECTED_DOC_PATH}')

    btn_up.on_click(on_up_clicked)
    btn_open.on_click(on_open_clicked)

    refresh_listing()
    display(path_label, listing, widgets.HBox([btn_up, btn_open]), status)
    print('راهنما: پوشه‌ها را انتخاب کنید و «📂 باز کردن / تأیید انتخاب» را بزنید.')


In [ ]:
#@title 🎙️ آپلود صدای مرجع برای کلون

from google.colab import files
import os, json, shutil, subprocess

#@markdown یک فایل صوتی **۳ تا ۱۰ ثانیه** با گفتار واضح و بدون نویز زمینه.
#@markdown صدای طولانی‌تر از ۱۰ ثانیه کیفیت کلونینگ را کاهش می‌دهد (به‌طور خودکار بریده می‌شود).
#@markdown اگر متن دقیقِ گفته‌شده در فایل را بنویسید، نتیجه بهتر و سریع‌تر است.
Ref_Text = '' #@param {type:"string"}
#@markdown ---
#@markdown #### 🗣️ تنظیمات تولید صدا (OmniVoice)
OmniVoice_Steps = 32 #@param {type:"slider", min:8, max:64, step:8}
OmniVoice_Speed = 1.0 #@param {type:"slider", min:0.5, max:2.0, step:0.1}
Random_Seed = 1234 #@param {type:"integer"}
Max_Ref_Seconds = 10 #@param {type:"slider", min:3, max:20, step:1}
Use_Text_Normalizer = INSTALL_TEXT_NORMALIZER #@param {type:"boolean"}

SR = 24000

if int(OmniVoice_Steps) < 16:
    print('⚠️ num_step کمتر از ۱۶ کیفیت را به‌شدت کاهش می‌دهد. ۳۲ توصیه می‌شود.')


def probe_duration(path):
    ff = shutil.which('ffprobe')
    if not ff:
        return None
    r = subprocess.run([ff, '-v', 'error', '-show_entries', 'format=duration',
                        '-of', 'json', path], capture_output=True, text=True)
    try:
        return float(json.loads(r.stdout)['format']['duration'])
    except Exception:
        return None


for f in ('ref_voice.wav', 'ref_voice.mp3', 'ref_voice.m4a'):
    if os.path.exists(f):
        os.remove(f)

print('📁 لطفاً فایل صوتی صدای مرجع را آپلود کنید:')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('❌ فایلی آپلود نشد!')

src = list(uploaded.keys())[0]
dur = probe_duration(src)

if dur is not None:
    print(f'⏱️ طول فایل آپلودی: {dur:.2f} ثانیه')
    if dur < 3:
        print('⚠️ صدا کوتاه‌تر از ۳ ثانیه است؛ کیفیت کلون پایین می‌آید.')
    if dur > Max_Ref_Seconds:
        print(f'✂️ فایل بلندتر از {Max_Ref_Seconds} ثانیه است؛ به {Max_Ref_Seconds} ثانیه بریده می‌شود.')

# تبدیل/نرمال‌سازی به wav مونو 24kHz (بدون دست‌کاری نام فایل در shell)
REF_AUDIO_PATH = 'ref_voice.wav'
cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-i', src,
       '-ar', str(SR), '-ac', '1']
if dur is not None and dur > Max_Ref_Seconds:
    cmd += ['-t', str(int(Max_Ref_Seconds))]
cmd.append(REF_AUDIO_PATH)
subprocess.run(cmd, check=True)

REF_TEXT = Ref_Text

print(f'✅ صدای مرجع ذخیره شد: {REF_AUDIO_PATH}')
print(f'📝 متن مرجع: {"[خودکار با Whisper]" if not REF_TEXT.strip() else REF_TEXT}')
print(f'🗣️ گام‌های دیفیوژن: {OmniVoice_Steps} | ⚡ سرعت: {OmniVoice_Speed}x | 🎲 seed: {Random_Seed}')
print(f'🔤 نرمال‌سازی متن: {"روشن" if Use_Text_Normalizer else "خاموش"}')


In [ ]:
#@title ▶️ شروع فرآیند: خواندن فایل با صدای کلون‌شده

import os, re, random, shutil, subprocess, traceback
import numpy as np
import torch, torchaudio
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, Audio

YOUTUBE_CHANNEL_URL = "https://www.youtube.com/@aigolden?sub_confirmation=1"

SUBTITLE_EXTS = ('.srt', '.vtt', '.sub', '.ass')
SR = 24000
SILENCE_MS = 350
OUTPUT_FILE = 'cloned_voice_docx_reading.mp3'
SEG_DIR = 'voice_clone_segments'
FAIL_FILE = 'failed_paragraphs.txt'


# ───────────────────────── استخراج متن
def _clean_subtitle(raw):
    out = []
    for block in re.split(r'\n\s*\n', raw):
        lines = []
        for line in block.splitlines():
            s = line.strip()
            if not s:
                continue
            if s.isdigit():                      # شماره‌ی بلوک
                continue
            if '-->' in s:                       # تایم‌کد
                continue
            if re.match(r'^(WEBVTT|NOTE\b|STYLE\b|REGION\b)', s, re.I):
                continue
            if re.match(r'^\d{1,2}:\d{2}:\d{2}[.,]\d{3}\s*$', s):
                continue
            lines.append(s)
        if lines:
            out.append(' '.join(lines))          # خطوط یک cue به هم می‌چسبند
    return out


def _iter_docx_blocks(doc):
    from docx.document import Document as _Doc
    from docx.oxml.table import CT_Tbl
    from docx.oxml.text.paragraph import CT_P
    from docx.table import Table, _Cell
    from docx.text.paragraph import Paragraph

    def walk(parent):
        if isinstance(parent, _Doc):
            elm = parent.element.body
        elif isinstance(parent, _Cell):
            elm = parent._tc
        else:
            return
        for child in elm.iterchildren():
            if isinstance(child, CT_P):
                yield Paragraph(child, parent)
            elif isinstance(child, CT_Tbl):
                yield Table(child, parent)

    yield from walk(doc)


def extract_paragraphs(path):
    ext = path.rsplit('.', 1)[-1].lower()

    if ext == 'docx':
        import docx
        from docx.table import Table

        doc = docx.Document(path)
        out = []

        for block in _iter_docx_blocks(doc):
            if isinstance(block, Table):
                for row in block.rows:
                    cells, seen = [], set()
                    for cell in row.cells:
                        if id(cell._tc) in seen:      # سلول‌های ادغام‌شده تکرار می‌شوند
                            continue
                        seen.add(id(cell._tc))
                        t = ' '.join(p.text.strip() for p in cell.paragraphs if p.text.strip())
                        if t:
                            cells.append(t)
                    if cells:
                        out.append(' ، '.join(cells))
            else:
                t = block.text.strip()
                if t:
                    out.append(t)
        return out

    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()

    if f'.{ext}' in SUBTITLE_EXTS:
        return _clean_subtitle(raw)

    # بلوک‌ها با خط خالی جدا می‌شوند؛ خطوطِ شکسته‌ی یک پاراگراف به هم می‌چسبند
    out = []
    for block in re.split(r'\n\s*\n', raw):
        lines = [l.strip() for l in block.splitlines() if l.strip()]
        if lines:
            out.append(' '.join(lines))
    return out


def merge_short(paras, min_len=25):
    merged = []
    for p in paras:
        if merged and len(p) < min_len:
            merged[-1] = merged[-1].rstrip() + ' ' + p
        else:
            merged.append(p)
    return merged


# ───────────────────────── تولید صدا
def synth(text, out_path):
    kwargs = dict(
        text=text,
        ref_audio=REF_AUDIO_PATH,
        num_step=int(OmniVoice_Steps),
        speed=float(OmniVoice_Speed),
    )
    if REF_TEXT.strip():
        kwargs['ref_text'] = REF_TEXT
    if Use_Text_Normalizer:
        kwargs['normalize_text'] = True

    audio = omni_model.generate(**kwargs)
    t = audio[0]
    if not isinstance(t, torch.Tensor):
        t = torch.tensor(np.asarray(t))
    if t.dim() == 1:
        t = t.unsqueeze(0)
    torchaudio.save(out_path, t.cpu(), SR)


def concat_segments(seg_paths, out_path, gap_ms=SILENCE_MS, work_dir=None):
    if not seg_paths:
        raise RuntimeError('هیچ سگمنتی تولید نشد!')

    work_dir = work_dir or SEG_DIR
    os.makedirs(work_dir, exist_ok=True)

    silence = os.path.join(work_dir, '_silence.wav')
    ff = shutil.which('ffmpeg')

    if ff:
        subprocess.run(
            [ff, '-y', '-loglevel', 'error', '-f', 'lavfi',
             '-i', f'anullsrc=r={SR}:cl=mono', '-t', str(gap_ms / 1000), silence],
            check=True,
        )
        lst = os.path.join(work_dir, '_list.txt')
        with open(lst, 'w', encoding='utf-8') as f:
            for i, p in enumerate(seg_paths):
                f.write(f"file '{os.path.abspath(p)}'\n")
                if i != len(seg_paths) - 1:
                    f.write(f"file '{os.path.abspath(silence)}'\n")

        r = subprocess.run(
            [ff, '-y', '-loglevel', 'error', '-f', 'concat', '-safe', '0',
             '-i', lst, '-c:a', 'libmp3lame', '-b:a', '192k', out_path],
            capture_output=True, text=True,
        )
        if r.returncode == 0 and os.path.exists(out_path):
            return out_path
        print(f'⚠️ ffmpeg ناموفق بود ({r.stderr.strip()[:200]}) — از pydub استفاده می‌کنم.')

    from pydub import AudioSegment
    final = AudioSegment.empty()
    for i, p in enumerate(seg_paths):
        final += AudioSegment.from_file(p)
        if i != len(seg_paths) - 1:
            final += AudioSegment.silent(duration=gap_ms)
    final.export(out_path, format='mp3', bitrate='192k')
    return out_path


# ───────────────────────── اجرا
if 'SELECTED_DOC_PATH' not in globals() or not SELECTED_DOC_PATH:
    raise RuntimeError('❌ ابتدا در سلول «انتخاب فایل متنی» یک فایل انتخاب کنید!')
if 'REF_AUDIO_PATH' not in globals() or not os.path.exists(REF_AUDIO_PATH):
    raise RuntimeError('❌ ابتدا سلول «آپلود صدای مرجع» را اجرا کنید!')

print(f'📄 در حال خواندن متن از: {SELECTED_DOC_PATH}')
PARAGRAPHS = merge_short(extract_paragraphs(SELECTED_DOC_PATH))

if not PARAGRAPHS:
    raise RuntimeError('❌ متنی در فایل یافت نشد!')

total_chars = sum(len(p) for p in PARAGRAPHS)
print(f'✅ {len(PARAGRAPHS)} پاراگراف استخراج شد ({total_chars} کاراکتر).')

os.makedirs(SEG_DIR, exist_ok=True)
for f in os.listdir(SEG_DIR):
    if f.startswith('seg_'):
        os.remove(os.path.join(SEG_DIR, f))

random.seed(int(Random_Seed))
np.random.seed(int(Random_Seed) % (2 ** 32))
torch.manual_seed(int(Random_Seed))
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(int(Random_Seed))

seg_paths, failures = [], []

for i, text in enumerate(PARAGRAPHS):
    seg = os.path.join(SEG_DIR, f'seg_{i+1:04d}.wav')
    preview = text[:40].replace('\n', ' ')

    for attempt in range(1, 4):
        try:
            synth(text, seg)
            if not os.path.exists(seg) or os.path.getsize(seg) < 1000:
                raise RuntimeError('فایل خروجی خالی است')
            seg_paths.append(seg)
            print(f'   ✅ [{i+1}/{len(PARAGRAPHS)}] {preview}...')
            break
        except Exception as e:
            if attempt == 3:
                failures.append((i + 1, text, repr(e)))
                print(f'   ❌ [{i+1}/{len(PARAGRAPHS)}] پس از ۳ تلاش رد شد: {e}')
                traceback.print_exc(limit=1)
            else:
                print(f'   ↻ [{i+1}] تلاش {attempt} ناموفق ({type(e).__name__}) — تلاش مجدد...')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
print(f'🎬 ساخت تراک نهایی از {len(seg_paths)} سگمنت ...')
output_file = concat_segments(seg_paths, OUTPUT_FILE)

if failures:
    with open(FAIL_FILE, 'w', encoding='utf-8') as f:
        for idx, txt, err in failures:
            f.write(f'[{idx}] {err}\n{txt}\n\n')
    print(f'⚠️ {len(failures)} پاراگراف خوانده نشد — جزئیات در {FAIL_FILE}')
    for idx, txt, err in failures[:5]:
        print(f'   • [{idx}] {txt[:60]}... → {err}')

size_mb = os.path.getsize(output_file) / 1e6
print(f'🎉 تراک نهایی آماده شد: {output_file} ({size_mb:.1f} MB)')


# ───────────────────────── پخش و دانلود
display(Audio(output_file, autoplay=True))

btn_download = widgets.Button(description='⬇️ دانلود تراک نهایی', button_style='success')
btn_download.on_click(lambda b: files.download(output_file))
display(btn_download)

if failures:
    btn_fail = widgets.Button(description='⬇️ دانلود لیست پاراگراف‌های ناموفق', button_style='warning')
    btn_fail.on_click(lambda b: files.download(FAIL_FILE))
    display(btn_fail)

display(HTML(
    f'<a href="{YOUTUBE_CHANNEL_URL}" target="_blank" '
    f'style="display:inline-block;margin-top:10px;padding:10px 22px;background:#FF0000;'
    f'color:#fff;border-radius:6px;text-decoration:none;font-family:sans-serif;font-weight:bold;">'
    f'🔔 ما را در یوتیوب دنبال کنید</a>'
))


In [ ]:
#@title ✏️ اصلاح دستی تلفظ (جایگزینی در فایل و در حافظه)

import os
import ipywidgets as widgets
from IPython.display import display

if 'SELECTED_DOC_PATH' not in globals() or not SELECTED_DOC_PATH or not os.path.exists(SELECTED_DOC_PATH):
    raise RuntimeError('❌ ابتدا سلول «انتخاب فایل متنی» را اجرا کنید!')

CURRENT_FILE = SELECTED_DOC_PATH
FILE_EXT = CURRENT_FILE.rsplit('.', 1)[-1].lower()


def _replace_in_paragraph(p, w, c):
    full = p.text
    n = full.count(w)
    if not n:
        return 0

    # اول در سطح run تا فرمت‌بندی (بولد/رنگ) حفظ شود
    hit = False
    for run in p.runs:
        if w in run.text:
            run.text = run.text.replace(w, c)
            hit = True

    # اگر کلمه بین چند run شکسته شده بود، ناچاراً کل پاراگراف بازنویسی می‌شود
    if not hit:
        for run in list(p.runs)[1:]:
            run.text = ''
        if p.runs:
            p.runs[0].text = full.replace(w, c)
    return n


def replace_in_file(w, c):
    total = 0

    if FILE_EXT == 'docx':
        import docx
        doc = docx.Document(CURRENT_FILE)
        for p in doc.paragraphs:
            total += _replace_in_paragraph(p, w, c)
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    for p in cell.paragraphs:
                        total += _replace_in_paragraph(p, w, c)
        if total:
            doc.save(CURRENT_FILE)
        return total

    with open(CURRENT_FILE, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.read().splitlines()

    new_lines = []
    for line in lines:
        total += line.count(w)
        new_lines.append(line.replace(w, c))

    if total:
        with open(CURRENT_FILE, 'w', encoding='utf-8') as f:
            f.write('\n'.join(new_lines))
    return total


wrong_word = widgets.Text(description='کلمه اشتباه:', placeholder='مثلاً: نکته',
                          layout=widgets.Layout(width='460px'))
correct_word = widgets.Text(description='عبارت درست:', placeholder='مثلاً: نُکته',
                            layout=widgets.Layout(width='460px'))
btn_replace = widgets.Button(description='🔁 جایگزینی', button_style='warning')
log_output = widgets.Output()

display(widgets.HTML(f'<b>📄 فایل فعلی:</b> {CURRENT_FILE}'),
        wrong_word, correct_word, btn_replace, log_output)


def on_replace_clicked(b):
    with log_output:
        log_output.clear_output()

        w = wrong_word.value.strip()
        c = correct_word.value.strip()

        if not w or not c:
            print('⚠️ هر دو فیلد را پر کنید.')
            return
        if w == c:
            print('⚠️ عبارت اشتباه و درست یکسان‌اند.')
            return

        n_file = replace_in_file(w, c)

        n_mem = 0
        if 'PARAGRAPHS' in globals() and PARAGRAPHS:
            for i, p in enumerate(PARAGRAPHS):
                if w in p:
                    n_mem += p.count(w)
                    PARAGRAPHS[i] = p.replace(w, c)

        if n_file == 0 and n_mem == 0:
            print(f'ℹ️ عبارت «{w}» پیدا نشد.')
            return

        print(f'✅ {n_file} مورد در فایل و {n_mem} مورد در حافظه جایگزین شد.')
        wrong_word.value = ''
        correct_word.value = ''


btn_replace.on_click(on_replace_clicked)

print('راهنما: بعد از گوش دادن، کلمه‌ی اشتباه‌خوانده‌شده را در فیلد اول و شکل درست (با اعراب/نیم‌فاصله) '
      'را در فیلد دوم بنویس و «جایگزینی» را بزن. برای هر اشتباه تکرار کن، سپس سلول «▶️ شروع فرآیند» '
      'را دوباره اجرا کن تا صدای اصلاح‌شده تولید شود.')


In [ ]:
#@title 🧪 (اختیاری) تست دستی بدون فایل — وارد کردن متن مستقیم

# نکته: این سلول از توابع تعریف‌شده در سلول «▶️ شروع فرآیند» استفاده می‌کند،
# پس یک بار آن سلول را اجرا کنید (حتی اگر فایل ندارید).

import os
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, Audio

Manual_Text = "" #@param {type:"string"}
YOUTUBE_CHANNEL_URL = "https://www.youtube.com/@aigolden?sub_confirmation=1"

if 'synth' not in globals():
    raise RuntimeError('❌ ابتدا یک بار سلول «▶️ شروع فرآیند» را اجرا کنید تا توابع آماده شوند.')
if 'REF_AUDIO_PATH' not in globals() or not os.path.exists(REF_AUDIO_PATH):
    raise RuntimeError('❌ ابتدا سلول «آپلود صدای مرجع» را اجرا کنید!')

paragraphs = [l.strip() for l in Manual_Text.splitlines() if l.strip()]
if not paragraphs:
    raise RuntimeError('❌ متنی وارد نشده است!')

MANUAL_SEG_DIR = 'voice_clone_segments_manual'
os.makedirs(MANUAL_SEG_DIR, exist_ok=True)

seg_paths, failures = [], []
for i, text in enumerate(paragraphs):
    seg = os.path.join(MANUAL_SEG_DIR, f'seg_{i+1:04d}.wav')
    for attempt in range(1, 4):
        try:
            synth(text, seg)
            if not os.path.exists(seg) or os.path.getsize(seg) < 1000:
                raise RuntimeError('فایل خروجی خالی است')
            seg_paths.append(seg)
            print(f'   ✅ [{i+1}/{len(paragraphs)}] {text[:40]}...')
            break
        except Exception as e:
            if attempt == 3:
                failures.append((i + 1, text, repr(e)))
                print(f'   ❌ [{i+1}] رد شد: {e}')
            else:
                print(f'   ↻ [{i+1}] تلاش {attempt} ناموفق — تلاش مجدد...')

import shutil
output_file = concat_segments(seg_paths, 'cloned_voice_manual_test.mp3', work_dir=MANUAL_SEG_DIR)

print(f'🎉 تراک آزمایشی آماده شد: {output_file}')
display(Audio(output_file, autoplay=True))

btn_download = widgets.Button(description='⬇️ دانلود تراک آزمایشی', button_style='success')
btn_download.on_click(lambda b: files.download(output_file))
display(btn_download)

display(HTML(
    f'<a href="{YOUTUBE_CHANNEL_URL}" target="_blank" '
    f'style="display:inline-block;margin-top:10px;padding:10px 22px;'
    f'background:#FF0000;color:#fff;border-radius:6px;'
    f'text-decoration:none;font-family:sans-serif;font-weight:bold;">'
    f'🔔 ما را در یوتیوب دنبال کنید</a>'
))


In [ ]:
#@title 🧹 پاکسازی فایل‌های موقت

import os, glob, shutil

#@markdown فایل‌های خروجی (mp3) و فایل Drive دست‌نخورده می‌مانند.
#@markdown گزینه‌ی دوم همه‌چیز جز Drive و خروجی‌های mp3 را حذف می‌کند — با احتیاط.
Wipe_Everything = False #@param {type:"boolean"}

KEEP_SUFFIXES = ('.mp3',)

removed = []

for pattern in ('/content/voice_clone_segments*', '/content/ref_voice.*',
                '/content/_silence.wav', '/content/_list.txt'):
    for p in glob.glob(pattern):
        if os.path.isdir(p):
            shutil.rmtree(p, ignore_errors=True)
        else:
            try:
                os.remove(p)
            except OSError:
                continue
        removed.append(p)

if Wipe_Everything:
    for entry in os.listdir('/content'):
        if entry == 'drive':
            continue
        if entry.lower().endswith(KEEP_SUFFIXES):
            continue
        full = os.path.join('/content', entry)
        try:
            shutil.rmtree(full) if os.path.isdir(full) else os.remove(full)
            removed.append(full)
        except OSError:
            pass

print('🧹 حذف شد:' if removed else 'ℹ️ چیزی برای حذف نبود.')
for p in removed:
    print('  •', p)


## ⚠️ نکات مهم

### تگ‌های صوتی غیرکلامی (بعضی‌ها ضعیف عمل می‌کنند)

| تگ | واکنش صوتی |
|---|---|
| `[laughter]` | خنده |
| `[sigh]` | آه کشیدن / نفس عمیق از خستگی |
| `[confirmation-en]` | صدای تاییدی (هوم / باشه) |
| `[question-en]` | «en» صدای سؤالی با مصوت |
| `[question-ah]` | «ah» صدای سؤالی با مصوت |
| `[question-oh]` | «oh» صدای سؤالی با مصوت |
| `[question-ei]` | «ei» صدای سؤالی با مصوت |
| `[question-yi]` | «yi» صدای سؤالی با مصوت |
| `[surprise-ah]` | «ah» صدای تعجبی با مصوت |
| `[surprise-oh]` | «oh» صدای تعجبی با مصوت |
| `[surprise-wa]` | «wa» صدای تعجبی با مصوت |
| `[surprise-yo]` | «yo» صدای تعجبی با مصوت |
| `[dissatisfaction-hnn]` | صدای نارضایتی («هنن») |

### راهنمای کیفیت

- **صدای مرجع:** ۳ تا ۱۰ ثانیه، تک‌گوینده، بدون موسیقی و نویز. طولانی‌تر = کلون ضعیف‌تر.
- **`num_step`:** ۳۲ پیش‌فرض و متعادل است؛ ۱۶ سریع‌تر با کیفیت کمتر؛ بالای ۳۲ معمولاً تفاوت محسوسی ندارد.
- **`Ref_Text`:** اگر متن دقیقِ صدای مرجع را بنویسید، Whisper لود نمی‌شود → هم سریع‌تر هم دقیق‌تر.
- **فارسی:** نرمال‌سازی متن روشن باشد تا اعداد («۱۲۳» → «صد و بیست و سه») درست خوانده شوند.
- **تلفظ غلط:** از سلول «✏️ اصلاح دستی تلفظ» استفاده کنید؛ با اعراب و نیم‌فاصله (مثل «نُکته»، «می‌رود») نتیجه بهتر است.
- **بازتولیدپذیری:** با `Random_Seed` ثابت، خروجی هر اجرا یکسان می‌ماند.

### تفاوت‌های این نسخه با نسخه‌ی قبلی

1. ترتیب نصب درست شد: `torch` + `torchaudio` با هم و قبل از `omnivoice`.
2. `ipywidgets` دیگر با `-U` آپدیت نمی‌شود (ویجت‌های Colab نمی‌شکنند).
3. محافظ GPU: روی CPU به‌طور پیش‌فرض اجرا نمی‌شود.
4. `normalize_text` برای خواندن درست اعداد.
5. پاراگراف‌بندی: خطوط wrap‌شده به هم می‌چسبند (قبلاً هر خط یک پاراگراف جدا بود).
6. جدول‌های docx هم خوانده می‌شوند (قبلاً کلاً نادیده گرفته می‌شدند).
7. زیرنویس‌های srt/vtt از تایم‌کد و شماره پاک می‌شوند.
8. سکوت اضافه بعد از آخرین پاراگراف حذف شد.
9. چسباندن با `ffmpeg` → مصرف حافظه‌ی ثابت حتی برای اسناد بلند.
10. خطاها با traceback گزارش و در `failed_paragraphs.txt` ذخیره می‌شوند.
11. `seed` برای خروجی بازتولیدپذیر.
12. سلول پاکسازی دیگر همه‌چیز را حذف نمی‌کند.
